In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit

In [2]:
source = pd.read_json("python_dataset.jsonl", lines=True)

In [3]:
df = source

df = df.head(1000)

df = df.drop(columns="docstring")

# Convert df from wide format to long format. Originally it was idx, human_code, gpt_code, qwen_code ...
# But i need seperate rows per code snippet. With melt I transform it to idx, code, source
df = pd.melt(
    df,
    id_vars="hm_index",
    value_vars=["human_code", "chatgpt_code", "dsc_code", "qwen_code"],
    var_name="source",
    value_name="code",
)

df = df.dropna(axis=0)

# Add new column thats 0 or 1 depending on source. Human-written is 0, AI-generated is 1
df["ai_generated"] = df["source"].ne("human_code").astype(int)

df = df.rename(columns={"hm_index": "task_id"})

# Simplify task_id values and sort df
df["task_id"] = df.groupby("task_id").ngroup() + 1
df = df.sort_values(by=["task_id"])

# Move colum order so source comes last
df = df.iloc[:, [0, 2, 3, 1]]


In [24]:
# Split code into input X and ouput y
X = df["code"]
y = df["ai_generated"]

# The df has multiple rows per coding task. e.g. task 1 has 4 rows (human-, gpt-, qwen- and dsc- code snippets)
# with GroupShuffleShift I split the the rows but dont put rows with the same task_id into train AND test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(df, groups=df["task_id"]):
    train_X, train_y = X.iloc[train_idx], y.iloc[train_idx]
    test_X, test_y = X.iloc[test_idx], y.iloc[test_idx]

# schneidung = train_X["task_id"].isin(test_X["task_id"]).any()
# print(schneidung)

# CountVectorizer:
# Takes documents and makes vectors with the count of the tokens as dimensions. Simple example: "hello cat. Bye cat." would become (1, 2, 1)
# Step 1 fit: finds all unique tokens and builds a vocabulary
# Step 2 transform: counts how many times the tokens appear and creates a vector per document

# TfidfTransformer:
# Takes count matrix (multiple vectors) and compares the frequency of each token with the num of appearances of that token in all the vectors.
# TF: How many times does this token appear in a document?
# IDF: In how many different documents does this token appear in?
# Step 1 fit: calculates the inverse document frequency / weights. So (num of documents / num of document where token appears)
# Step 2 transform: Looks at a single document and multiplies the tokens in the count vector with
# the corresponding idf weight calculated in Step 1. Forms and normalises a vector.


# Init a count vectorizer. make a count vector for each row in the training data (only code)
cv = CountVectorizer()
train_count = cv.fit_transform(train_X)

# Init a tfidf transformer. pass all the count vectors into transformer and get a vector for each row in the training data
transformer = TfidfTransformer()
train_tfidf = transformer.fit_transform(train_count)


# make test tfidf. only use transform to avoid data leakage
test_count = cv.transform(test_X)
test_tfidf = transformer.transform(test_count)


In [ ]:
# Training the model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42,
)

model = model.fit(train_tfidf, train_y)

pred_y = model.predict(test_tfidf)

accuracy = accuracy_score(test_y, pred_y)
accuracy


0.8175